# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AxelYoel/FlyRank-AI-Internship---Axel-Yoel-Chandra/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [18]:

import duckdb, os
from google.colab import userdata

hf_token = userdata.get("HF_token")
os.environ["HF_token"] = hf_token

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")


In [3]:
rel = "hf://datasets/FlyRank/internship-warehouse"

# --- 0. Confirm the exact column names first, don't assume ---
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet')").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [4]:
# --- 1. Staleness distribution: days since last content update ---
# Using the same window-end date (2026-06-30) on w03,
# and your own filters: published, not deleted.
con.sql(f"""
    SELECT
        MIN(days_since_update)                              AS min_days,
        approx_quantile(days_since_update, 0.25)             AS p25,
        approx_quantile(days_since_update, 0.50)             AS median,
        approx_quantile(days_since_update, 0.75)             AS p75,
        approx_quantile(days_since_update, 0.90)             AS p90,
        MAX(days_since_update)                              AS max_days
    FROM (
        SELECT content_hash_id,
               DATE_DIFF('day', content_updated_date, DATE '2026-06-30') AS days_since_update
        FROM read_parquet('{rel}/dim_content.parquet')
        WHERE is_published IS TRUE AND is_deleted IS NOT TRUE
    )
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_days,p25,median,p75,p90,max_days
0,-6,18,41,41,107,394


In [5]:
# --- A1. How many rows are affected, and how negative do they get? ---
con.sql(f"""
    SELECT
        COUNT(*) AS n_negative_rows,
        MIN(days_since_update) AS most_negative,
        MAX(days_since_update) AS least_negative
    FROM (
        SELECT content_hash_id,
               DATE_DIFF('day', content_updated_date, DATE '2026-06-30') AS days_since_update
        FROM read_parquet('{rel}/dim_content.parquet')
        WHERE is_published IS TRUE AND is_deleted IS NOT TRUE
    )
    WHERE days_since_update < 0
""").df()

,n_negative_rows,most_negative,least_negative
0,69629,-6,-1


In [6]:
# --- A2. Look at a sample of the actual rows — is content_updated_date genuinely in the future? ---
con.sql(f"""
    SELECT content_hash_id, content_created_date, content_updated_date,
           DATE_DIFF('day', content_updated_date, DATE '2026-06-30') AS days_since_update
    FROM read_parquet('{rel}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS NOT TRUE
      AND DATE_DIFF('day', content_updated_date, DATE '2026-06-30') < 0
    ORDER BY content_updated_date DESC
    LIMIT 20
""").df()

,content_hash_id,content_created_date,content_updated_date,days_since_update
0,content_0d7b1fdad154eb7b,2026-07-06,2026-07-06,-6
1,content_17667347cd331f5d,2026-07-06,2026-07-06,-6
2,content_1f77541e81009c1b,2026-07-06,2026-07-06,-6
3,content_215f34c8fab2b277,2026-07-06,2026-07-06,-6
4,content_4307a0ff4f47cd0c,2026-07-06,2026-07-06,-6
5,content_4a78755ddfed26e5,2026-07-06,2026-07-06,-6
6,content_6009654d5567d7cf,2026-07-06,2026-07-06,-6
7,content_7151324a09c0e11a,2026-07-06,2026-07-06,-6
8,content_721c3e1fedd8671b,2026-07-06,2026-07-06,-6
9,content_76192bed978feece,2026-07-06,2026-07-06,-6


In [7]:
# --- B1. Does days_since_update actually cluster at specific values, like optimization_eligible_date did? ---
con.sql(f"""
    SELECT days_since_update, COUNT(*) AS n_pages
    FROM (
        SELECT content_hash_id,
               DATE_DIFF('day', content_updated_date, DATE '2026-06-30') AS days_since_update
        FROM read_parquet('{rel}/dim_content.parquet')
        WHERE is_published IS TRUE AND is_deleted IS NOT TRUE
    )
    GROUP BY 1
    ORDER BY n_pages DESC
    LIMIT 15
""").df()

,days_since_update,n_pages
0,41,202374
1,-1,38314
2,125,32287
3,29,24130
4,-3,18034
5,43,12914
6,13,12029
7,-4,5644
8,18,4882
9,-5,4717


In [8]:
con.sql(f"""
    SELECT content_created_date, COUNT(*) AS n_pages
    FROM read_parquet('{rel}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS NOT TRUE
    GROUP BY 1
    ORDER BY n_pages DESC
    LIMIT 15
""").df()

,content_created_date,n_pages
0,2025-06-27,7915
1,2026-04-23,6640
2,2025-07-14,6033
3,2025-09-12,5902
4,2025-04-20,5757
5,2025-07-31,5571
6,2025-07-11,4643
7,2025-09-19,4396
8,2025-05-30,4207
9,2026-01-16,4127


In [9]:
con.sql(f"""
    SELECT client_hash_id,
           COUNT(*) AS n_pages_total,
           SUM(CASE WHEN DATE_DIFF('day', content_updated_date, DATE '2026-06-30') = 41 THEN 1 ELSE 0 END) AS n_pages_in_cluster
    FROM read_parquet('{rel}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS NOT TRUE
    GROUP BY 1
    ORDER BY n_pages_in_cluster DESC
    LIMIT 20
""").df()


,client_hash_id,n_pages_total,n_pages_in_cluster
0,client_625b6439094e23e4,31887,30209.0
1,client_3ffa76342f366962,32595,27954.0
2,client_08a6a72ff48e62c0,29809,14917.0
3,client_ba65e80a1116ae41,13126,12282.0
4,client_2b4306c3ed003f01,12126,11292.0
5,client_65de48885f4ef01b,13493,10435.0
6,client_3197e6291363b4db,11171,6645.0
7,client_73cda7b4e4f265ea,30970,6307.0
8,client_a80fca3f171ed1de,7624,6183.0
9,client_cd12bcfd98942aa1,12526,5364.0


In [10]:
# --- 2. Visibility distribution: total impressions per page over the 90-day window ---
# Fixed: fact_content_daily_performance is partitioned by month, so it needs a glob, not a single filename.
con.sql(f"""
    SELECT
        MIN(total_impr)                          AS min_impr,
        approx_quantile(total_impr, 0.25)         AS p25,
        approx_quantile(total_impr, 0.50)         AS median,
        approx_quantile(total_impr, 0.75)         AS p75,
        approx_quantile(total_impr, 0.90)         AS p90,
        MAX(total_impr)                          AS max_impr
    FROM (
        SELECT content_hash_id, SUM(gsc_impressions) AS total_impr
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE gsc_data_available IS TRUE
          AND report_date BETWEEN DATE '2026-04-02' AND DATE '2026-06-30'
        GROUP BY 1
    )
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_impr,p25,median,p75,p90,max_impr
0,1.0,16.0,168.0,1256.0,5828.0,1961517.0


In [11]:
# --- 3. Pool size check: how many pages would pass at a few candidate threshold combos ---
con.sql(f"""
    WITH staleness AS (
        SELECT content_hash_id,
               DATE_DIFF('day', content_updated_date, DATE '2026-06-30') AS days_since_update
        FROM read_parquet('{rel}/dim_content.parquet')
        WHERE is_published IS TRUE AND is_deleted IS NOT TRUE
    ),
    visibility AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS total_impr
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE gsc_data_available IS TRUE
          AND report_date BETWEEN DATE '2026-04-02' AND DATE '2026-06-30'
        GROUP BY 1
    )
    SELECT
        SUM(CASE WHEN days_since_update >= 180 AND total_impr >= 100 THEN 1 ELSE 0 END) AS pass_180d_100impr,
        SUM(CASE WHEN days_since_update >= 180 AND total_impr >= 500 THEN 1 ELSE 0 END) AS pass_180d_500impr,
        SUM(CASE WHEN days_since_update >= 365 AND total_impr >= 100 THEN 1 ELSE 0 END) AS pass_365d_100impr,
        SUM(CASE WHEN days_since_update >= 365 AND total_impr >= 500 THEN 1 ELSE 0 END) AS pass_365d_500impr,
        COUNT(*) AS total_eligible_denominator
    FROM staleness s
    JOIN visibility v USING (content_hash_id)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pass_180d_100impr,pass_180d_500impr,pass_365d_100impr,pass_365d_500impr,total_eligible_denominator
0,280.0,102.0,0.0,0.0,260250


In [12]:
# Age distribution + check for the same "created after window end" issue we found before
con.sql(f"""
    SELECT
        MIN(age_days) AS min_age,
        approx_quantile(age_days, 0.25) AS p25,
        approx_quantile(age_days, 0.50) AS median,
        approx_quantile(age_days, 0.75) AS p75,
        approx_quantile(age_days, 0.90) AS p90,
        MAX(age_days) AS max_age,
        SUM(CASE WHEN age_days < 0 THEN 1 ELSE 0 END) AS n_negative
    FROM (
        SELECT content_hash_id,
               DATE_DIFF('day', content_created_date, DATE '2026-06-30') AS age_days
        FROM read_parquet('{rel}/dim_content.parquet')
        WHERE is_published IS TRUE AND is_deleted IS NOT TRUE
    )
""").df()

,min_age,p25,median,p75,p90,max_age,n_negative
0,-6,116,284,354,439,585,3855.0


In [13]:
# Aggregate query-level prev30 fields up to page-level, then check the distribution
con.sql(f"""
    WITH page_level AS (
        SELECT
            content_hash_id,
            SUM(impressions_prev30) AS impressions_prev30_page,
            SUM(avg_position_prev30 * impressions_prev30) / NULLIF(SUM(impressions_prev30), 0) AS avg_position_prev30_page
        FROM read_parquet('{rel}/fact_content_query_90d.parquet')
        GROUP BY 1
    )
    SELECT
        COUNT(*) AS n_pages_with_query_data,
        SUM(CASE WHEN impressions_prev30_page = 0 THEN 1 ELSE 0 END) AS n_zero_impr,
        approx_quantile(impressions_prev30_page, 0.50) AS median_impr,
        approx_quantile(impressions_prev30_page, 0.90) AS p90_impr,
        approx_quantile(avg_position_prev30_page, 0.50) AS median_pos,
        approx_quantile(avg_position_prev30_page, 0.90) AS p90_pos,
        MIN(avg_position_prev30_page) AS min_pos,
        MAX(avg_position_prev30_page) AS max_pos
    FROM page_level
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_pages_with_query_data,n_zero_impr,median_impr,p90_impr,median_pos,p90_pos,min_pos,max_pos
0,133852,13666.0,49.0,915.0,21.40761,71.145425,0.0,405.0


In [19]:
# How common is the position=0 pattern, and does it look like the same "missing" convention?
con.sql(f"""
    SELECT
        COUNT(*) AS n_query_rows,
        SUM(CASE WHEN avg_position_prev30 = 0 THEN 1 ELSE 0 END) AS n_zero_position_rows,
        SUM(CASE WHEN avg_position_prev30 = 0 AND impressions_prev30 = 0 THEN 1 ELSE 0 END) AS n_zero_position_and_zero_impr
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_query_rows,n_zero_position_rows,n_zero_position_and_zero_impr
0,2414248,65922.0,0.0


In [14]:
# Coverage check: does every eligible page even have query-level data,
# or does the query table only cover a subset (pages with ranked queries in that period)?
con.sql(f"""
    WITH eligible_pages AS (
        SELECT content_hash_id FROM read_parquet('{rel}/dim_content.parquet')
        WHERE is_published IS TRUE AND is_deleted IS NOT TRUE
    ),
    query_pages AS (
        SELECT DISTINCT content_hash_id FROM read_parquet('{rel}/fact_content_query_90d.parquet')
    )
    SELECT
        (SELECT COUNT(*) FROM eligible_pages) AS n_eligible_total,
        (SELECT COUNT(*) FROM eligible_pages e JOIN query_pages q USING (content_hash_id)) AS n_with_query_coverage
""").df()

,n_eligible_total,n_with_query_coverage
0,411540,133801


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.